# Predictor encoding and predictive validity

This study compares prediction on five encodings of the same binary modifier:
string labels, integer `0/1`, booleans, signed `+1/−1` and shifted `2/3` values.
Each encoding has separate known-specification and data-alone discovery tracks.
Four tree libraries tune on development data, and Ridge is the simple
reference. The table shows the proposed binary-reference repair; the baseline,
regressions and remaining shifted-coding gaps are retained below.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from great_tables import GT
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(ROOT))
from benches.report_encoding_tracks import DIRECTORY, gap_table, headline

baseline = json.loads((DIRECTORY / "baseline/summary.json").read_text())
after = json.loads((DIRECTORY / "after/summary.json").read_text())
display(headline(after))


GT(_tbl_data=                  track             method                     categorical  \
0   known_specification     mboostrs_known  <strong>1.026</strong> (0.017)   
1   known_specification           catboost                   1.218 (0.020)   
2   known_specification           lightgbm                   1.417 (0.029)   
3   known_specification            xgboost                   1.559 (0.054)   
4   known_specification             histgb                   1.405 (0.021)   
5   known_specification              ridge                   3.035 (0.039)   
6             discovery  mboostrs_discover  <strong>1.084</strong> (0.021)   
7             discovery           catboost                   1.218 (0.020)   
8             discovery           lightgbm                   1.417 (0.029)   
9             discovery            xgboost                   1.559 (0.054)   
10            discovery             histgb                   1.405 (0.021)   
11            discovery              ridge                   3.035 (0.039)   

                         indicator                         boolean  \
0   <strong>1.026</strong> (0.017)  <strong>1.026</strong> (0.017)   
1                    1.216 (0.023)                   1.216 (0.023)   
2                    1.417 (0.029)                   1.417 (0.029)   
3                    1.559 (0.054)                   1.559 (0.054)   
4                    1.405 (0.021)                   1.405 (0.021)   
5                    3.035 (0.039)                   3.035 (0.039)   
6   <strong>1.084</strong> (0.021)  <strong>1.084</strong> (0.021)   
7                    1.216 (0.023)                   1.216 (0.023)   
8                    1.417 (0.029)                   1.417 (0.029)   
9                    1.559 (0.054)                   1.559 (0.054)   
10                   1.405 (0.021)                   1.405 (0.021)   
11                   3.035 (0.039)                   3.035 (0.039)   

                            signed                         shifted  
0   <strong>1.012</strong> (0.017)                   1.349 (0.026)  
1                    1.216 (0.023)  <strong>1.216</strong> (0.023)  
2                    1.417 (0.029)                   1.417 (0.029)  
3                    1.559 (0.054)                   1.559 (0.054)  
4                    1.405 (0.021)                   1.405 (0.021)  
5                    3.035 (0.039)                   3.035 (0.039)  
6   <strong>1.077</strong> (0.023)                   1.386 (0.030)  
7                    1.216 (0.023)  <strong>1.216</strong> (0.023)  
8                    1.417 (0.029)                   1.417 (0.029)  
9                    1.559 (0.054)                   1.559 (0.054)  
10                   1.405 (0.021)                   1.405 (0.021)  
11                   3.035 (0.039)                   3.035 (0.039)  , _body=<great_tables._gt_data.Body object at 0x10faa7c20>, _boxhead=Boxhead([ColInfo(var='track', type=<ColInfoTypeEnum.row_group: 3>, column_label='track', column_align='left', column_width=None), ColInfo(var='method', type=<ColInfoTypeEnum.stub: 2>, column_label='method', column_align='left', column_width=None), ColInfo(var='categorical', type=<ColInfoTypeEnum.default: 1>, column_label='categorical', column_align='left', column_width=None), ColInfo(var='indicator', type=<ColInfoTypeEnum.default: 1>, column_label='indicator', column_align='left', column_width=None), ColInfo(var='boolean', type=<ColInfoTypeEnum.default: 1>, column_label='boolean', column_align='left', column_width=None), ColInfo(var='signed', type=<ColInfoTypeEnum.default: 1>, column_label='signed', column_align='left', column_width=None), ColInfo(var='shifted', type=<ColInfoTypeEnum.default: 1>, column_label='shifted', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1101139b0>, _spanners=Spanners([SpannerInfo(spanner_id='Encoding', spanner_level=0, spanner_label='Encoding', spanner_units=None, spanner_pattern=None, vars=['categorical', 'i

## Paired encoding views

The five views reuse exactly the same eight latent data draws and responses.
They are not forty independent samples. Each draw has 800 development rows and
2,000 independent test rows. Competitors train on the first 600 development
rows, select one of four configurations by MSE on the next 200, then refit on
all 800. Both mboostrs tracks use all 800 with five internal folds. Their
search designs and fit counts differ, so this study makes no matched-compute
claim.

Five numeric predictors are independent Uniform(−1,1). An independent binary
variable `b` determines the response mean
`2*sin(pi*x1)*(1−2*b)+0.5*x2`; independent Gaussian noise has standard deviation
1 response unit. Discovery receives only the data and response name, plus
seed, folds and a 1,600-update ceiling. Known specification receives the
corresponding true term structure and Gaussian family. Supplied terms receive
no discovery credit.

For string, `0/1`, boolean and shifted encodings the known dictionary has a
baseline `bbs(x1)` plus `bbs(x1, by=g)` and `bols(x2)`. With signed coding,
`bbs(x1, by=g)+bols(x2)` directly spans the true mean. In shifted coding,
writing `f(x)=2*sin(pi*x)` gives `5*f(x)−2*g*f(x)` at `g=2+b`. The dictionary
is valid, but its large cancelling components can require a longer boosting
path. A finite-iteration loss here is an estimation result, not a missing-term
error in the supplied formula. The [protocol](encoding_tracks/PROTOCOL.md)
fixes all formulas, seeds, grids, losses, score rules and sample sizes.


## Gaps to competing methods

CatBoost has the lowest mean test MSE among the four complete tree competitors
for every encoding. This reference is selected retrospectively on test scores.
The primary gap averages eight paired ratios, `100*(mboost_MSE/peer_MSE−1)`;
its SE is their sample standard deviation divided by the square root of eight.
It conditions on the selected peer and omits winner-selection uncertainty.
Every ± quantity below denotes one Monte Carlo SE, not a confidence bound.

Known-specification gaps are −15.69% ± 0.87 points for strings,
−15.49% ± 1.20 for both `0/1` and boolean inputs, and −16.64% ± 1.29 for
signed inputs. Discovery gaps after repair are −10.90% ± 1.56, −10.68% ± 1.82
for both `0/1` and booleans, and −11.25% ± 1.89 for signed inputs. Those eight
track/encoding cells meet the at-most-1% target. Shifted coding remains above
it: +10.99% ± 1.33 in known specification and +14.12% ± 2.01 in discovery.
All encodings and both failures to meet target remain in the table.


In [2]:
display(gap_table(after))


GT(_tbl_data=                 track         task best_competitor  completed    gap_pct  \
0  known_specification  categorical        catboost          8 -15.685775   
1            discovery  categorical        catboost          8 -10.898238   
2  known_specification    indicator        catboost          8 -15.489031   
3            discovery    indicator        catboost          8 -10.683606   
4  known_specification      boolean        catboost          8 -15.489031   
5            discovery      boolean        catboost          8 -10.683606   
6  known_specification       signed        catboost          8 -16.641200   
7            discovery       signed        catboost          8 -11.249244   
8  known_specification      shifted        catboost          8  10.993129   
9            discovery      shifted        catboost          8  14.122563   

   gap_se_pct  difference_mse  difference_se  within_1pct  unresolved_at_2se  
0    0.870130       -0.191392       0.012023         True              False  
1    1.560126       -0.133571       0.020191         True              False  
2    1.198482       -0.189346       0.017180         True              False  
3    1.821401       -0.131526       0.024436         True              False  
4    1.198482       -0.189346       0.017180         True              False  
5    1.821401       -0.131526       0.024436         True              False  
6    1.289523       -0.203460       0.018634         True              False  
7    1.893359       -0.138296       0.025422         True              False  
8    1.334492        0.132915       0.015551        False              False  
9    2.010351        0.170697       0.023489        False              False  , _body=<great_tables._gt_data.Body object at 0x10a27fc20>, _boxhead=Boxhead([ColInfo(var='track', type=<ColInfoTypeEnum.row_group: 3>, column_label='track', column_align='left', column_width=None), ColInfo(var='task', type=<ColInfoTypeEnum.stub: 2>, column_label='task', column_align='left', column_width=None), ColInfo(var='best_competitor', type=<ColInfoTypeEnum.default: 1>, column_label='best_competitor', column_align='left', column_width=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='completed', column_align='right', column_width=None), ColInfo(var='gap_pct', type=<ColInfoTypeEnum.default: 1>, column_label='gap_pct', column_align='right', column_width=None), ColInfo(var='gap_se_pct', type=<ColInfoTypeEnum.default: 1>, column_label='gap_se_pct', column_align='right', column_width=None), ColInfo(var='difference_mse', type=<ColInfoTypeEnum.default: 1>, column_label='difference_mse', column_align='right', column_width=None), ColInfo(var='difference_se', type=<ColInfoTypeEnum.default: 1>, column_label='difference_se', column_align='right', column_width=None), ColInfo(var='within_1pct', type=<ColInfoTypeEnum.default: 1>, column_label='within_1pct', column_align='center', column_width=None), ColInfo(var='unresolved_at_2se', type=<ColInfoTypeEnum.default: 1>, column_label='unresolved_at_2se', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10c091ac0>, _spanners=Spanners([SpannerInfo(spanner_id='Gap (%)', spanner_level=0, spanner_label='Gap (%)', spanner_units=None, spanner_pattern=None, vars=['gap_pct', 'gap_se_pct'], built=None), SpannerInfo(spanner_id='Paired MSE difference', spanner_level=0, spanner_label='Paired MSE difference', spanner_units=None, spanner_pattern=None, vars=['difference_mse', 'difference_se'], built=None)]), _heading=Heading(title='Encoding tracks: paired gap to the strongest complete named competitor', subtitle='Percent gap and paired Monte Carlo SE; negative favors mboostrs; target at most +1%', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1101a1d60>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1101a0740>, _source_notes=['The competitor is selected retrospectively fr

## Binary reference repair

The parent already contains the string-factor repair, but numeric and boolean
partners were excluded from its reference-completion rule. A by term is zero
at `g=0`. Opposite curves at `0` and `1` therefore need a continuous baseline
when their marginal mean cancels and elimination removes it. The repair in
[issue #572](https://github.com/fcela/mboostrs/issues/572), with its separate fix in [MR #575](https://github.com/fcela/mboostrs/pull/575) makes that baseline
available for binary partners while preserving their original values, existing
terms, input scale, constraints and adoption threshold.

For `0/1` and booleans, discovery's MSE falls from 2.07415 to 1.08423. The paired
percent change is −47.69% ± 0.94 points; its gap to CatBoost changes from
+70.84% ± 2.87 to −10.68% ± 1.82. Their repaired predictions match the string
view byte-for-byte on all eight draws. All 40 known fits and 200 competitor/Ridge
predictions remain exact across phases. String discovery is unchanged.

Signed discovery has a retained small regression: mean MSE rises by 0.40%
with paired SE 0.53 points. This is unresolved at this precision and remains
within the competitor target. Shifted discovery improves by 54.46% ± 0.46
points, but still misses target; a large relative gain does not erase its
remaining +14.12% gap.


In [3]:
changes = json.loads((DIRECTORY / "paired_improvement.json").read_text())
frame = pd.DataFrame(changes["changes"])
display(GT(frame, rowname_col="task", groupname_col="track")
    .tab_header(title="Same observations and protocol before and after binary repair",
        subtitle="Mean test MSE, paired change and SE; negative change improves prediction")
    .tab_spanner("Prediction", ["before_mse", "after_mse"])
    .tab_spanner("Paired change", ["change_mse", "change_se", "change_pct", "change_pct_se"])
    .fmt_number(columns=["before_mse", "after_mse", "change_mse", "change_se"], decimals=4)
    .fmt_number(columns=["change_pct", "change_pct_se"], decimals=2)
    .tab_source_note("Each row uses eight matching latent draws. SE=sample SD of paired changes/sqrt(8); encodings are not pooled.")
    .tab_source_note("Categorical predictions and all known/peer predictions replay exactly. Signed and shifted losses are retained."))


GT(_tbl_data=          task                track  completed  before_mse  after_mse  \
0  categorical  known_specification          8    1.026406   1.026406   
1  categorical            discovery          8    1.084227   1.084227   
2    indicator  known_specification          8    1.026406   1.026406   
3    indicator            discovery          8    2.074146   1.084227   
4      boolean  known_specification          8    1.026406   1.026406   
5      boolean            discovery          8    2.074146   1.084227   
6       signed  known_specification          8    1.012292   1.012292   
7       signed            discovery          8    1.073377   1.077457   
8      shifted  known_specification          8    1.348667   1.348667   
9      shifted            discovery          8    3.042688   1.386449   

   change_mse  change_se  change_pct  change_pct_se  unchanged_predictions  
0    0.000000   0.000000    0.000000       0.000000                      8  
1    0.000000   0.000000    0.000000       0.000000                      8  
2    0.000000   0.000000    0.000000       0.000000                      8  
3   -0.989919   0.028398  -47.688287       0.938837                      0  
4    0.000000   0.000000    0.000000       0.000000                      8  
5   -0.989919   0.028398  -47.688287       0.938837                      0  
6    0.000000   0.000000    0.000000       0.000000                      8  
7    0.004079   0.005526    0.401277       0.528857                      0  
8    0.000000   0.000000    0.000000       0.000000                      8  
9   -1.656238   0.016310  -54.462187       0.459659                      0  , _body=<great_tables._gt_data.Body object at 0x10faa7c20>, _boxhead=Boxhead([ColInfo(var='task', type=<ColInfoTypeEnum.stub: 2>, column_label='task', column_align='left', column_width=None), ColInfo(var='track', type=<ColInfoTypeEnum.row_group: 3>, column_label='track', column_align='left', column_width=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='completed', column_align='right', column_width=None), ColInfo(var='before_mse', type=<ColInfoTypeEnum.default: 1>, column_label='before_mse', column_align='right', column_width=None), ColInfo(var='after_mse', type=<ColInfoTypeEnum.default: 1>, column_label='after_mse', column_align='right', column_width=None), ColInfo(var='change_mse', type=<ColInfoTypeEnum.default: 1>, column_label='change_mse', column_align='right', column_width=None), ColInfo(var='change_se', type=<ColInfoTypeEnum.default: 1>, column_label='change_se', column_align='right', column_width=None), ColInfo(var='change_pct', type=<ColInfoTypeEnum.default: 1>, column_label='change_pct', column_align='right', column_width=None), ColInfo(var='change_pct_se', type=<ColInfoTypeEnum.default: 1>, column_label='change_pct_se', column_align='right', column_width=None), ColInfo(var='unchanged_predictions', type=<ColInfoTypeEnum.default: 1>, column_label='unchanged_predictions', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1101a0c20>, _spanners=Spanners([SpannerInfo(spanner_id='Prediction', spanner_level=0, spanner_label='Prediction', spanner_units=None, spanner_pattern=None, vars=['before_mse', 'after_mse'], built=None), SpannerInfo(spanner_id='Paired change', spanner_level=0, spanner_label='Paired change', spanner_units=None, spanner_pattern=None, vars=['change_mse', 'change_se', 'change_pct', 'change_pct_se'], built=None)]), _heading=Heading(title='Same observations and protocol before and after binary repair', subtitle='Mean test MSE, paired change and SE; negative change improves prediction', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1101a2330>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1101a2150>, _source_notes=['Each row uses eight matching latent draws. SE=sample SD of paired changes/sqrt(8); encodings are not pooled.', 'Categorical pre

## Fresh controls

A [separate frozen control battery](binary_reference/PROTOCOL.md) uses six new
draws in each of nine families: categorical, numeric, boolean, signed and
shifted opposite curves; an active-only curve; an additive smooth; pure noise;
and a log-scale numeric interaction. All 108 phase/draw jobs complete. Every
family's mean after/before MSE ratio is at most 1.01; gains in one family do
not compensate for a failing family. Numeric and boolean zero-side curves
become nonconstant, categorical predictions are exact, and null interaction
adoptions do not increase. Active, smooth and null predictions are unchanged.

These are controls against the parent procedure, not a best-competitor target
or calibrated false-discovery claim. Signed controls pass even though the
separate primary signed mean rises slightly; both results are reported.


In [4]:
from benches.report_binary_reference import control_table

controls = json.loads((ROOT / "docs/book/benchmarks/binary_reference/summary.json").read_text())
display(control_table(controls))


GT(_tbl_data=          case  completed  before_mse  after_mse     ratio    change  \
0  categorical          6    0.298347   0.298347  1.000000  0.000000   
1    indicator          6    1.108710   0.289926  0.261499 -0.818783   
2      boolean          6    1.111889   0.285277  0.256570 -0.826612   
3       signed          6    0.315410   0.312205  0.989839 -0.003205   
4      shifted          6    2.307039   0.974397  0.422358 -1.332642   
5       active          6    0.256905   0.256905  1.000000  0.000000   
6       smooth          6    0.252253   0.252253  1.000000  0.000000   
7         null          6    0.252115   0.252115  1.000000  0.000000   
8       logged          6    1.159477   0.317169  0.273545 -0.842308   

   change_se  passes_1_01  
0   0.000000         True  
1   0.164253         True  
2   0.165725         True  
3   0.003831         True  
4   0.016896         True  
5   0.000000         True  
6   0.000000         True  
7   0.000000         True  
8   0.168960         True  , _body=<great_tables._gt_data.Body object at 0x10a2cfda0>, _boxhead=Boxhead([ColInfo(var='case', type=<ColInfoTypeEnum.stub: 2>, column_label='case', column_align='left', column_width=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='completed', column_align='right', column_width=None), ColInfo(var='before_mse', type=<ColInfoTypeEnum.default: 1>, column_label='before_mse', column_align='right', column_width=None), ColInfo(var='after_mse', type=<ColInfoTypeEnum.default: 1>, column_label='after_mse', column_align='right', column_width=None), ColInfo(var='ratio', type=<ColInfoTypeEnum.default: 1>, column_label='ratio', column_align='right', column_width=None), ColInfo(var='change', type=<ColInfoTypeEnum.default: 1>, column_label='change', column_align='right', column_width=None), ColInfo(var='change_se', type=<ColInfoTypeEnum.default: 1>, column_label='change_se', column_align='right', column_width=None), ColInfo(var='passes_1_01', type=<ColInfoTypeEnum.default: 1>, column_label='passes_1_01', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1101a33b0>, _spanners=Spanners([SpannerInfo(spanner_id='Prediction', spanner_level=0, spanner_label='Prediction', spanner_units=None, spanner_pattern=None, vars=['before_mse', 'after_mse', 'ratio'], built=None), SpannerInfo(spanner_id='Paired change', spanner_level=0, spanner_label='Paired change', spanner_units=None, spanner_pattern=None, vars=['change', 'change_se'], built=None)]), _heading=Heading(title='Fresh controls for the binary-reference dictionary', subtitle='Test MSE, after/before mean ratio and paired change SE; lower is better; limit 1.01', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1101a39e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1101a3920>, _source_notes=['Six independent new draws per family. SE is sample SD of the paired after-minus-before MSE divided by sqrt(6). Every family must pass; improvements do not compensate regressions.', 'This is a parent non-regression gate, not a best-competitor target or calibrated FDR claim. Null-adoption and exact categorical prediction checks are separate gates.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1101a3860>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1101a3020>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=Opt

## Shifted coding and stopping range

[Issue #573](https://github.com/fcela/mboostrs/issues/573) records the shifted gap with a complete
dictionary. All eight known optima and all eight repaired discovery
optima remain at 1,599 or 1,600 updates. The additive baseline and scalar-by
blocks are strongly correlated under `2/3` coding; complete representation
does not imply that this coordinate path has converged within the fixed budget.

A [frozen diagnostic](encoding_tracks/conditioning/PROTOCOL.md) keeps the known
formula, data, folds, step and response score unchanged and raises only the
search ceiling to 25,600. All sixteen original indicator/shifted prefixes
replay exactly. Indicator MSE stays 1.02641; shifted MSE falls from 1.34867 to
1.03182, moving its gap from +10.99% to −15.05% against the recorded CatBoost
reference. Shifted selected counts range from 8,917 to 22,398 updates. This
supports a finite-path explanation for the known shifted result. It does not
retroactively change the original budget or select a new default from test loss.


In [5]:
diagnostic = json.loads((DIRECTORY / "conditioning/summary.json").read_text())
frame = pd.DataFrame(diagnostic["groups"])
frame["mstop_by_seed"] = frame["mstop_by_seed"].map(lambda values: ", ".join(map(str, values)))
display(GT(frame, rowname_col="pool", groupname_col="task")
    .tab_header(title="Known-specification stopping-range diagnostic",
        subtitle="Test MSE and paired gap percent; lower is better; longer searches are diagnostic")
    .tab_spanner("Prediction", ["mean_mse", "mse_se", "gap_pct", "gap_se_pct"])
    .fmt_number(columns=["mean_mse", "mse_se"], decimals=4)
    .fmt_number(columns=["gap_pct", "gap_se_pct"], decimals=2)
    .tab_source_note("All eight draws per encoding. Sixteen prefix models exactly replay the original known fits. Reference is retrospective CatBoost with unequal search budget.")
    .tab_source_note("The longer ceiling selects no default; the original 1,600-step results remain the primary benchmark."))


GT(_tbl_data=        task           pool  completed  mean_mse    mse_se    gap_pct  \
0  indicator     prefix1600          8  1.026406  0.017391 -15.489031   
1  indicator  longer_search          8  1.026406  0.017391 -15.489031   
2    shifted     prefix1600          8  1.348667  0.025546  10.993129   
3    shifted  longer_search          8  1.031822  0.017427 -15.046996   

   gap_se_pct                                      mstop_by_seed  
0    1.198482            613, 714, 630, 764, 619, 718, 775, 1170  
1    1.198482            613, 714, 630, 764, 619, 718, 775, 1170  
2    1.334492     1599, 1600, 1599, 1599, 1600, 1600, 1600, 1600  
3    1.160242  9314, 11192, 9649, 9762, 11106, 8917, 10241, 2...  , _body=<great_tables._gt_data.Body object at 0x10a2cfda0>, _boxhead=Boxhead([ColInfo(var='task', type=<ColInfoTypeEnum.row_group: 3>, column_label='task', column_align='left', column_width=None), ColInfo(var='pool', type=<ColInfoTypeEnum.stub: 2>, column_label='pool', column_align='left', column_width=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='completed', column_align='right', column_width=None), ColInfo(var='mean_mse', type=<ColInfoTypeEnum.default: 1>, column_label='mean_mse', column_align='right', column_width=None), ColInfo(var='mse_se', type=<ColInfoTypeEnum.default: 1>, column_label='mse_se', column_align='right', column_width=None), ColInfo(var='gap_pct', type=<ColInfoTypeEnum.default: 1>, column_label='gap_pct', column_align='right', column_width=None), ColInfo(var='gap_se_pct', type=<ColInfoTypeEnum.default: 1>, column_label='gap_se_pct', column_align='right', column_width=None), ColInfo(var='mstop_by_seed', type=<ColInfoTypeEnum.default: 1>, column_label='mstop_by_seed', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1101a1d60>, _spanners=Spanners([SpannerInfo(spanner_id='Prediction', spanner_level=0, spanner_label='Prediction', spanner_units=None, spanner_pattern=None, vars=['mean_mse', 'mse_se', 'gap_pct', 'gap_se_pct'], built=None)]), _heading=Heading(title='Known-specification stopping-range diagnostic', subtitle='Test MSE and paired gap percent; lower is better; longer searches are diagnostic', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1101a2810>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1101a0a10>, _source_notes=['All eight draws per encoding. Sixteen prefix models exactly replay the original known fits. Reference is retrospective CatBoost with unequal search budget.', 'The longer ceiling selects no default; the original 1,600-step results remain the primary benchmark.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1101a3fe0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10a4dbf50>, <great_tables._gt_data.FormatInfo object at 0x1101a3f80>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight

```{figure} encoding_tracks/conditioning/cv_paths.png
:name: encoding-cv-paths

Known-specification cross-validation MSE paths for eight paired latent draws.
Solid blue paths use `0/1`; dashed orange paths use `2/3`. The dotted vertical
line marks the original 1,600-update limit. Each line ends at its own explored
grid; these are individual paths, not a confidence band or a pooled mean.
The longer searches are diagnostic. A [vector export](encoding_tracks/conditioning/cv_paths.svg)
is available.
```


## Discovery stopping ceiling

The diagnostic also exposed [issue #574](https://github.com/fcela/mboostrs/issues/574), repaired separately in [MR #576](https://github.com/fcela/mboostrs/pull/576).
Discovery passed a growing start grid to the tuner but omitted the requested
maximum. The public tuner's default 10,000 ceiling then clipped a larger start,
while discovery could report a grid of 12,800 or 25,600. This bug does not
explain the original 1,600-step losses, since that budget is below the hidden
ceiling.

The separate repair forwards the effective maximum to tuning and preflight.
Eight public shifted calls requesting 25,600 have a declared/actual mismatch
on the parent and none after repair. The corrected searches reach an actual
12,800-position grid and turn there; five selected optima exceed 10,000.
All 80 original known/discovery fits at the unchanged 1,600 limit replay exact
inputs, predictions and MSE. This checks whether the requested and actual stopping ceilings agree; competitor
measurements are not relabelled as new fits, and the original ten benchmark
cells are unchanged.

The [contract protocol](encoding_tracks/cap_contract/PROTOCOL.md) fixes those
calls. Its first observer missed a warning decorator and recorded no stages;
that [incomplete capture](encoding_tracks/cap_contract/initial_observer/before_high.json)
is retained and was not treated as an audit pass. The corrected observer walks
the wrapper stack and rejects a successful high-cap row without actual returns.
All 96 numerical captures replay their first inputs/predictions/MSE exactly;
these repeats are not new independent confirmation.


In [6]:
cap = json.loads((DIRECTORY / "cap_contract/summary.json").read_text())
display(GT(pd.DataFrame(cap["high_rows"]), rowname_col="seed")
    .tab_header(title="Discovery maximum-grid contract: every shifted draw",
        subtitle="Actual update counts and test MSE; the requested maximum is 25,600 in both phases")
    .tab_spanner("Selected count", ["before_stop", "after_stop"])
    .tab_spanner("Test MSE", ["before_mse", "after_mse"])
    .tab_spanner("Observed search", ["before_mismatch", "after_mismatch", "after_maximum_forwarded", "before_largest", "after_largest"])
    .fmt_number(columns=["before_mse", "after_mse"], decimals=4)
    .tab_source_note("The maximum is a ceiling, not a requirement to continue after the CV minimum turns. Actual grid ends now match declarations.")
    .tab_source_note("Only the argument-propagation contract changes. Original 1,600-limit predictions are independently replayed on the cap-fixed package."))


GT(_tbl_data=   seed  before_stop  after_stop  before_mse  after_mse  before_mismatch  \
0     0         9266        9266    1.040142   1.040142             True   
1     1         9999       11771    1.040483   1.040800             True   
2     2         9997       11800    1.085786   1.084557             True   
3     3         8769        8769    1.075661   1.075661             True   
4     4         9999       10724    1.117771   1.117116             True   
5     5         9274        9274    1.057453   1.057453             True   
6     6         9915       10416    1.013702   1.014018             True   
7     7        10000       11451    1.214073   1.213285             True   

   after_mismatch  after_maximum_forwarded  before_largest  after_largest  
0           False                     True           10000          12800  
1           False                     True           10000          12800  
2           False                     True           10000          12800  
3           False                     True           10000          12800  
4           False                     True           10000          12800  
5           False                     True           10000          12800  
6           False                     True           10000          12800  
7           False                     True           10000          12800  , _body=<great_tables._gt_data.Body object at 0x10a27fc20>, _boxhead=Boxhead([ColInfo(var='seed', type=<ColInfoTypeEnum.stub: 2>, column_label='seed', column_align='right', column_width=None), ColInfo(var='before_stop', type=<ColInfoTypeEnum.default: 1>, column_label='before_stop', column_align='right', column_width=None), ColInfo(var='after_stop', type=<ColInfoTypeEnum.default: 1>, column_label='after_stop', column_align='right', column_width=None), ColInfo(var='before_mse', type=<ColInfoTypeEnum.default: 1>, column_label='before_mse', column_align='right', column_width=None), ColInfo(var='after_mse', type=<ColInfoTypeEnum.default: 1>, column_label='after_mse', column_align='right', column_width=None), ColInfo(var='before_mismatch', type=<ColInfoTypeEnum.default: 1>, column_label='before_mismatch', column_align='center', column_width=None), ColInfo(var='after_mismatch', type=<ColInfoTypeEnum.default: 1>, column_label='after_mismatch', column_align='center', column_width=None), ColInfo(var='after_maximum_forwarded', type=<ColInfoTypeEnum.default: 1>, column_label='after_maximum_forwarded', column_align='center', column_width=None), ColInfo(var='before_largest', type=<ColInfoTypeEnum.default: 1>, column_label='before_largest', column_align='right', column_width=None), ColInfo(var='after_largest', type=<ColInfoTypeEnum.default: 1>, column_label='after_largest', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10e9c8d70>, _spanners=Spanners([SpannerInfo(spanner_id='Selected count', spanner_level=0, spanner_label='Selected count', spanner_units=None, spanner_pattern=None, vars=['before_stop', 'after_stop'], built=None), SpannerInfo(spanner_id='Test MSE', spanner_level=0, spanner_label='Test MSE', spanner_units=None, spanner_pattern=None, vars=['before_mse', 'after_mse'], built=None), SpannerInfo(spanner_id='Observed search', spanner_level=0, spanner_label='Observed search', spanner_units=None, spanner_pattern=None, vars=['before_mismatch', 'after_mismatch', 'after_maximum_forwarded', 'before_largest', 'after_largest'], built=None)]), _heading=Heading(title='Discovery maximum-grid contract: every shifted draw', subtitle='Actual update counts and test MSE; the requested maximum is 25,600 in both phases', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1101e4bc0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1101e4200>, _source_notes=['The maximum is a ceiling, not a requirement to continue after the CV minimum turns. Actual grid ends now match declarations.', 'Only the ar

## Scorecard profile

The study measures D11 only, separately in ten encoding/track profiles. All
repaired cells score 3 under the frozen rule: useful outer prediction improves
on prespecified Ridge by more than two paired SE. The shifted discovery parent
scores 1 for failing to beat that reference; its repair reaches 3 while still
losing to CatBoost. This distinction separates usefulness from the 1% target.

Levels 4 and 5 are unavailable with these finite grids, unequal search designs
and narrow generator family. Encoding views do not constitute the full D6
robustness battery, and predictive loss supplies no recovery, explanation or
calibration grade. Eleven dimensions remain unmeasured per profile, with no
aggregate or cross-study average.


In [7]:
card = json.loads((DIRECTORY / "after/scorecard.json").read_text())
rows = [{"track": p["track"], "encoding": p["task"],
         "D11": next(d["score"] for d in p["dimensions"] if d["id"] == "D11"),
         "other_dimensions": "D1a, D1b, D2–D10: unmeasured", "aggregate": "none"}
        for p in card["profiles"]]
display(GT(pd.DataFrame(rows), rowname_col="encoding", groupname_col="track")
    .tab_header(title="Scorecard 2.1: predictive-validity profiles",
        subtitle="Anchored D11 grades 1–5, higher is better; other dimensions receive no number")
    .tab_source_note("The complete twelve-dimension profiles and evidence are retained in after/scorecard.json. No grade is inferred from the cap contract or longer-path diagnostic."))


GT(_tbl_data=                 track     encoding  D11              other_dimensions  \
0  known_specification  categorical    3  D1a, D1b, D2–D10: unmeasured   
1            discovery  categorical    3  D1a, D1b, D2–D10: unmeasured   
2  known_specification    indicator    3  D1a, D1b, D2–D10: unmeasured   
3            discovery    indicator    3  D1a, D1b, D2–D10: unmeasured   
4  known_specification      boolean    3  D1a, D1b, D2–D10: unmeasured   
5            discovery      boolean    3  D1a, D1b, D2–D10: unmeasured   
6  known_specification       signed    3  D1a, D1b, D2–D10: unmeasured   
7            discovery       signed    3  D1a, D1b, D2–D10: unmeasured   
8  known_specification      shifted    3  D1a, D1b, D2–D10: unmeasured   
9            discovery      shifted    3  D1a, D1b, D2–D10: unmeasured   

  aggregate  
0      none  
1      none  
2      none  
3      none  
4      none  
5      none  
6      none  
7      none  
8      none  
9      none  , _body=<great_tables._gt_data.Body object at 0x10a2cfda0>, _boxhead=Boxhead([ColInfo(var='track', type=<ColInfoTypeEnum.row_group: 3>, column_label='track', column_align='left', column_width=None), ColInfo(var='encoding', type=<ColInfoTypeEnum.stub: 2>, column_label='encoding', column_align='left', column_width=None), ColInfo(var='D11', type=<ColInfoTypeEnum.default: 1>, column_label='D11', column_align='right', column_width=None), ColInfo(var='other_dimensions', type=<ColInfoTypeEnum.default: 1>, column_label='other_dimensions', column_align='left', column_width=None), ColInfo(var='aggregate', type=<ColInfoTypeEnum.default: 1>, column_label='aggregate', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10a4dbf80>, _spanners=Spanners([]), _heading=Heading(title='Scorecard 2.1: predictive-validity profiles', subtitle='Anchored D11 grades 1–5, higher is better; other dimensions receive no number', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1101e67b0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1101e6ae0>, _source_notes=['The complete twelve-dimension profiles and evidence are retained in after/scorecard.json. No grade is inferred from the cap contract or longer-path diagnostic.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1101e6ab0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type

## Reproduction and limits

The [baseline](encoding_tracks/baseline.json) and [after](encoding_tracks/after.json)
retain all 280 planned outcomes each, the encoded input and common latent-row
hashes, predictions, candidate grids, full logs, warnings, source/native
identities and versions. The [paired changes](encoding_tracks/paired_improvement.json)
and [validation record](encoding_tracks/VALIDATION.md) retain counts and checks.
No earlier predictive_tracks or factor_reference result is overwritten.

Run `python -m benches.bench_encoding_tracks --phase baseline` using the recorded
categorical-only parent on `PYTHONPATH`, and `--phase after` with the binary
repair. Rebuild tables with `python -m benches.report_encoding_tracks --phase ...`.
The separate conditioning and cap-contract runners/reporters reproduce their
own frozen diagnostics. Native code is unchanged from the matching main
`056d475` release build. The notebook stores its executed outputs as required
by the benchmark guide.

These are measured proposals under review, not an assertion that an installed
release contains them. Eight of ten original-budget cells meet the 1% point-gap
target; the two shifted gaps remain #573. Longer paths and the repaired
high-cap API demonstrate an available remedy on these known draws, with a
different resource budget and no held-back-family confirmation. Eleven other
scorecard dimensions remain unmeasured here; the broader benchmark goal stays
active.
